# SegmentAE Example: Unlabeled Anomaly Detection

This notebook demonstrates how to use SegmentAE for unsupervised anomaly detection on unlabeled data, simulating real-world scenarios where ground truth is unavailable.

## Features demonstrated:
- Unsupervised anomaly detection workflow
- Built-in DenseAutoencoder for reconstruction
- Reconstruction error analysis
- Cluster-wise anomaly detection
- Optional ground truth evaluation

## Import Dependencies

In [ ]:
import warnings

import tensorflow as tf

from segmentae import Clustering, Preprocessing, SegmentAE
from segmentae.autoencoders import DenseAutoencoder  # , BatchNormAutoencoder
from segmentae.data_sources import load_dataset

warnings.filterwarnings("ignore")

## Data Loading (Unlabeled Scenario)

**Important**: In this example, it is simulated an unlabeled scenario by ignoring the target column during training. In real applications, you wouldn't have access to these labels.

In [ ]:
train, future_data, target = load_dataset(
    dataset_selection="htru2_dataset",  # Options: 'default_credit_card', 'htru2_dataset', 'shuttle_148'
    split_ratio=0.75,
)

# Reset indices
train = train.reset_index(drop=True)
future_data = future_data.reset_index(drop=True)

# For unlabeled scenario, it is not used the target column
X_train = train.drop(columns=[target]).copy()
X_future_data = future_data.drop(columns=[target]).copy()

print(f" Dataset: HTRU2 Pulsar Detection (Unlabeled Mode)")
print(f"  Training samples: {len(X_train)}")
print(f"  Future samples: {len(X_future_data)}")
print(f"  Features: {X_train.shape[1]}")
print(f"\n Target labels hidden (unsupervised scenario)")

# Save true labels for optional evaluation (in real scenario, these wouldn't exist)
# y_train = train[target].astype(int)
# y_future_data = future_data[target].astype(int)

## Data Preprocessing

In [ ]:
pr = Preprocessing(
    encoder="IFrequencyEncoder",  # "LabelEncoder", "OneHotEncoder"
    scaler="StandardScaler",  # "MinMaxScaler", "RobustScaler"
    imputer=None,
)

pr.fit(X=X_train)
X_train = pr.transform(X=X_train)
X_future_data = pr.transform(X=X_future_data)

print("✓ Preprocessing completed")
print(f"  Processed training shape: {X_train.shape}")
print(f"  Processed future shape: {X_future_data.shape}")

## Unsupervised Clustering

Apply clustering to discover natural data segments without using labels.

In [ ]:
# KMeans clustering for hard assignments
cl_model = Clustering(
    cluster_model=["KMeans"],  # "KMeans", "MiniBatchKMeans", "GMM", "Agglomerative"
    n_clusters=3,
    covariance_type="full",  # GMM-specific parameter
)

cl_model.clustering_fit(X=X_train)
print(f"✓ Clustering completed: {cl_model.n_clusters} clusters identified")

# Analyze cluster distribution
import pandas as pd
cluster_labels = cl_model.cluster_model[0].predict(X_train)
cluster_dist = pd.Series(cluster_labels).value_counts().sort_index()
print(f"\n Cluster distribution:")
for cluster_id, count in cluster_dist.items():
    percentage = count / len(X_train) * 100
    print(f"  Cluster {cluster_id}: {count:,} samples ({percentage:.1f}%)")

## Autoencoder Implementation for Unlabeled Data

In [ ]:
print(f"Num GPUs Available: {len(tf.config.experimental.list_physical_devices('GPU'))}")

# DenseAutoencoder with architecture optimized for reconstruction
denseAutoencoder = DenseAutoencoder(
    hidden_dims=[16, 12, 8, 4],  # Bottleneck architecture
    encoder_activation="relu",
    decoder_activation="relu",
    optimizer="adam",
    learning_rate=0.001,
    epochs=150,
    val_size=0.15,  # Validation split for monitoring
    stopping_patient=10,  # Early stopping
    dropout_rate=0.1,
    batch_size=None,
)

print("🔧 Autoencoder Configuration:")
print(f"  Architecture: {' → '.join(map(str, denseAutoencoder.hidden_dims))}")
print(f"  Epochs: {denseAutoencoder.epochs}")
print(f"  Learning rate: {denseAutoencoder.learning_rate}")
print(f"  Dropout rate: {denseAutoencoder.dropout_rate}")

In [ ]:
# Train autoencoder on unlabeled data
print("\n Training autoencoder on unlabeled data...")
denseAutoencoder.fit(input_data=X_train)
print("✓ Autoencoder training completed")

## Model Architecture and Training Analysis

In [ ]:
# Display model information
print(" Model Architecture:")
denseAutoencoder.summary()

print("\n Training History:")
denseAutoencoder.plot_training_loss()

## SegmentAE Integration for Unlabeled Anomaly Detection

In [ ]:
sg = SegmentAE(ae_model=denseAutoencoder, cl_model=cl_model)

# Train Reconstruction (Unsupervised)
sg.reconstruction(
    input_data=X_train,
    threshold_metric="mse",  # || "mse", "mae", "rmse", "max_error" - Can also use MAE, RMSE, or MAX_ERROR
)

print("✓ SegmentAE reconstruction phase completed")

## Reconstruction Error Analysis

Analyze the reconstruction errors to understand the baseline performance.

In [ ]:
# Access reconstruction metadata
preds_train = sg.preds_train
recon_metrics_train = sg.reconstruction_eval

print("=" * 50)
print(" RECONSTRUCTION ANALYSIS")
print("=" * 50)
print(f"Reconstruction threshold: {sg.threshold:.6f}")
print(f"Mean reconstruction error: {recon_metrics_train['mse'].mean():.6f}")
print(f"Std reconstruction error: {recon_metrics_train['mse'].std():.6f}")
print(f"Min reconstruction error: {recon_metrics_train['mse'].min():.6f}")
print(f"Max reconstruction error: {recon_metrics_train['mse'].max():.6f}")

# Analyze errors by cluster
cluster_recon_stats = recon_metrics_train.groupby('Cluster')['mse'].agg(['mean', 'std', 'count'])
print(f"\n Reconstruction Error by Cluster:")
print(cluster_recon_stats.round(6))

## Anomaly Detection on Unlabeled Future Data

Apply the trained model to detect anomalies in new, unlabeled data.

In [ ]:
# Test different threshold ratios
threshold_ratios = [1.0, 1.5, 2.0, 2.5, 3.0]

print(" Testing different threshold ratios:")
print("=" * 50)

ratio_results = []
for threshold_ratio in threshold_ratios:
    predictions = sg.detections(input_data=X_future_data, threshold_ratio=threshold_ratio)
    
    n_anomalies = predictions["Predicted Anomalies"].sum()
    anomaly_rate = n_anomalies / len(predictions)
    
    ratio_results.append({
        'Threshold_Ratio': threshold_ratio,
        'Anomalies_Detected': n_anomalies,
        'Anomaly_Rate': anomaly_rate
    })
    
    print(f"Threshold {threshold_ratio}: {n_anomalies:,} anomalies ({anomaly_rate:.1%})")

# Convert to DataFrame for easier analysis
results_df = pd.DataFrame(ratio_results)
print(f"\n Threshold Analysis:")
print(results_df.to_string(index=False))

## Detailed Analysis with Chosen Threshold

Perform detailed analysis using a moderate threshold ratio.

In [ ]:
# Use threshold ratio of 2.0 for detailed analysis
chosen_threshold = 2.0
final_predictions = sg.detections(input_data=X_future_data, threshold_ratio=chosen_threshold)

# Analyze Detection Results
n_anomalies = final_predictions["Predicted Anomalies"].sum()
anomaly_rate = n_anomalies / len(final_predictions)

print("=" * 50)
print(" ANOMALY DETECTION RESULTS")
print("=" * 50)
print(f"Chosen threshold ratio: {chosen_threshold}")
print(f"Anomalies detected: {n_anomalies:,} / {len(final_predictions):,}")
print(f"Anomaly rate: {anomaly_rate:.2%}")

## Cluster-wise Anomaly Analysis

In [ ]:
# Cluster-wise analysis
cluster_analysis = final_predictions.groupby("Cluster")["Predicted Anomalies"].agg(
    ["count", "sum", "mean"]
)
cluster_analysis.columns = ["Total Samples", "Anomalies", "Anomaly Rate"]

print("\n Cluster-wise Analysis:")
print("=" * 40)
print(cluster_analysis)

# Identify which clusters have highest anomaly rates
highest_anomaly_cluster = cluster_analysis['Anomaly Rate'].idxmax()
lowest_anomaly_cluster = cluster_analysis['Anomaly Rate'].idxmin()

print(f"\n Key Insights:")
print(f"  Cluster {highest_anomaly_cluster} has the highest anomaly rate: {cluster_analysis.loc[highest_anomaly_cluster, 'Anomaly Rate']:.1%}")
print(f"  Cluster {lowest_anomaly_cluster} has the lowest anomaly rate: {cluster_analysis.loc[lowest_anomaly_cluster, 'Anomaly Rate']:.1%}")

## Reconstruction Error Distribution Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get reconstruction errors for future data
future_recon_errors = sg.ae_model.predict(X_future_data)
future_mse = np.mean((X_future_data - future_recon_errors) ** 2, axis=1)

plt.figure(figsize=(12, 4))

# Plot 1: Reconstruction error distribution
plt.subplot(1, 3, 1)
plt.hist(future_mse, bins=50, alpha=0.7, color='skyblue')
plt.axvline(sg.threshold * chosen_threshold, color='red', linestyle='--', 
           label=f'Threshold (ratio={chosen_threshold})')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Frequency')
plt.title('Reconstruction Error Distribution')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Anomaly detection results
plt.subplot(1, 3, 2)
anomaly_labels = ['Normal', 'Anomaly']
anomaly_counts = [len(final_predictions) - n_anomalies, n_anomalies]
colors = ['lightgreen', 'salmon']
plt.pie(anomaly_counts, labels=anomaly_labels, colors=colors, autopct='%1.1f%%')
plt.title('Anomaly Detection Results')

# Plot 3: Cluster-wise anomaly rates
plt.subplot(1, 3, 3)
cluster_rates = cluster_analysis['Anomaly Rate'] * 100
bars = plt.bar(cluster_rates.index, cluster_rates.values, 
              color=['lightcoral' if rate > cluster_rates.mean() else 'lightblue' 
                     for rate in cluster_rates.values])
plt.xlabel('Cluster')
plt.ylabel('Anomaly Rate (%)')
plt.title('Anomaly Rate by Cluster')
plt.grid(True, alpha=0.3)

# Add value labels on bars
for bar, rate in zip(bars, cluster_rates.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{rate:.1f}%', ha='center')

plt.tight_layout()
plt.show()

print("\n Visualizations show:")
print("  1. Distribution of reconstruction errors with threshold")
print("  2. Overall proportion of detected anomalies")
print("  3. Cluster-specific anomaly rates")

## Export Results

Save the anomaly detection results for further analysis.

In [ ]:
# Prepare export data
export_data = final_predictions.copy()
export_data['Reconstruction_Error'] = future_mse
export_data['Threshold_Used'] = sg.threshold * chosen_threshold
export_data['Detection_Date'] = pd.Timestamp.now()

# Export to CSV (uncomment to save)
# export_data.to_csv('unlabeled_anomaly_results.csv', index=False)
# print("✓ Results exported to 'unlabeled_anomaly_results.csv'")

print(" Export Data Preview:")
print(export_data[['Cluster', 'Predicted Anomalies', 'Reconstruction_Error']].head(10))

print(f"\n Summary Statistics:")
print(f"  Total samples processed: {len(export_data):,}")
print(f"  Anomalies detected: {export_data['Predicted Anomalies'].sum():,}")
print(f"  Mean reconstruction error: {export_data['Reconstruction_Error'].mean():.6f}")
print(f"  Threshold used: {export_data['Threshold_Used'].iloc[0]:.6f}")

## Optional: Ground Truth Evaluation

**Note**: This section would not be available in a real unlabeled scenario. It's included here for educational purposes to evaluate the unsupervised approach.

In [ ]:
# Uncomment this section if you want to evaluate against ground truth
# (This would not be available in a real unlabeled scenario)

"""
from segmentae.metrics import metrics_classification

# Get ground truth labels
y_future_data = future_data[target].astype(int)

actual_metrics = metrics_classification(
    y_true=y_future_data,
    y_pred=final_predictions['Predicted Anomalies']
)

print("\n GROUND TRUTH EVALUATION (Educational Only)")
print("=" * 50)
print(" This would NOT be available in real unlabeled scenarios!")
print("\nActual Performance:")
for metric, value in actual_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Compare detected vs actual anomaly rates
actual_anomaly_rate = y_future_data.mean()
detected_anomaly_rate = final_predictions['Predicted Anomalies'].mean()

print(f"\n Anomaly Rate Comparison:")
print(f"  Actual anomaly rate: {actual_anomaly_rate:.2%}")
print(f"  Detected anomaly rate: {detected_anomaly_rate:.2%}")
print(f"  Difference: {abs(actual_anomaly_rate - detected_anomaly_rate):.2%}")
"""

print("\n In real applications:")
print("  - You would not have ground truth labels")
print("  - Evaluation would be based on domain expertise")
print("  - Consider investigating detected anomalies manually")
print("  - Monitor threshold performance over time")
print("  - Adjust thresholds based on operational feedback")